In [25]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
import matplotlib.animation as animation
from scipy.ndimage import gaussian_filter
import os
from scipy.sparse import lil_matrix, kron, identity, diags, hstack, vstack, identity
import scipy.sparse as sparse
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.cm as cm
import time
import datetime


In [20]:
######### FUNCIONES DE INTEGRACIÓN TEMPORAL Y EVOLUCIÓN #########
def RK4_FD(eq, fields, parameters, grids, dt, Nt, operators, t_rate):
    now = datetime.datetime.now()
    print(f'Hora de Inicio: {now.hour}:{now.minute}:{now.second}')
    time_init = time.time()

    t_grid = grids[0]
    x_grid = grids[1]
    y_grid = grids[2]

    fields_history = []
    time_grid = []

    for i in range(Nt - 1):
        t = t_grid[i]
        old_fields = fields.copy()

        k_1 = equations_FD(eq, old_fields, t, x_grid, y_grid, parameters, operators)
        k_2 = equations_FD(eq, old_fields + 0.5 * dt * k_1, t + 0.5 * dt, x_grid, y_grid, parameters, operators)
        k_3 = equations_FD(eq, old_fields + 0.5 * dt * k_2, t + 0.5 * dt, x_grid, y_grid, parameters, operators)
        k_4 = equations_FD(eq, old_fields + dt * k_3, t + dt, x_grid, y_grid, parameters, operators)

        new_fields = old_fields + dt * (k_1 + 2 * k_2 + 2 * k_3 + k_4) / 6
        fields = new_fields

        if i % t_rate == 0:
            fields_history.append(fields.copy())
            time_grid.append(t)
    now = datetime.datetime.now()
    print(f'Hora de Término: {now.hour}:{now.minute}:{now.second}')
    print(f"{time.time() - time_init} seg")

    return fields, fields_history, time_grid

def equations_FD(eq, field_slices, t_i, x_grid, y_grid, parameters, operators):
    if eq == 'wave_equation_1D':
        U_1 = field_slices[0]
        U_2 = field_slices[1]
        DD = operators[0]
        c = parameters[0]

        ddU_x = Der(DD, U_1)

        F = U_2
        G = c ** 2 * (ddU_x)

        fields = np.array([F, G])
    elif eq == 'wildfire':

        U_1 = field_slices[0]
        U_2 = field_slices[1]
        omega = parameters["omega"]
        beta = parameters["beta"]
        gamma = parameters["gamma"]
        alpha = parameters["alpha"]
        delta = parameters["delta"]
        eta = parameters["eta"]

        L = operators[0]
        K_road = operators[1]

        U1_safe = np.maximum(U_1, 1e-8)
        ignite_funct = np.exp(-(omega / U1_safe)) * (U_1 >= omega)
        transport = K_road.dot(U_1)

        F = -gamma * U_1 + beta * (L.dot(U_1)) + transport + alpha * U_2 * ignite_funct
        G = -delta * U_2 * ignite_funct

        fields = np.array([F, G])
    return fields

def sparse_DD_neumann(Nx, dx):
    data = np.ones((3, Nx))
    data[1] = -2 * data[1]
    diags = [-1, 0, 1]
    D2 = sparse.spdiags(data, diags, Nx, Nx) / (dx ** 2)
    D2 = sparse.lil_matrix(D2)
    # Condiciones de borde de Neumann: ajustar la primera y última fila
    D2[0, 0] = -2 / dx ** 2
    D2[0, 1] = 2 / dx ** 2

    D2[-1, -1] = -2 / dx ** 2
    D2[-1, -2] = 2 / dx ** 2
    return D2.tocsr()

In [21]:
######### ORDEN DE MAGNITUD PARAMETROS REALES (NO NECESARIO CORRER, NO HACE NADA) #########
D_phys = 1e-3  # m^2/s -> difusividad térmica efectiva (conducción + radiación + mezcla aire)
gamma_phys = 1e-3  # 1/s -> tasa de enfriamiento (pérdida de calor al ambiente)
delta_phys = 0.1  # 1/s -> tasa de consumo de combustible (qué tan rápido se quema Y)
alpha_phys = 20.0  # K/s -> generación de temperatura por combustión (energía liberada → calor)
eta_phys = 1e-3  # 1/s -> intensidad del transporte por carretera
omega_phys = 200.0  # K -> temperatura de ignición (umbral para que comience la combustión)

L0 = 10.0  # m
tau0 = 100.0  # s
T0 = 100.0  # K

#beta = D_phys * tau0 / L0 ** 2
#gamma = gamma_phys * tau0
#delta = delta_phys * tau0
#alpha = alpha_phys * tau0 / T0
#eta = eta_phys * tau0
#omega = omega_phys / T0

In [22]:
######### RUTINA PRINCIPAL #########
# ---------------- parámetros ----------------
project_name = '/wildfire_roads'
disc = 'D:/'
eq = 'wildfire'


beta = 0.005  # difusión
gamma = 0.1  # disipación
delta = 0.1  # consumo combustible
alpha = 3  # generación calor
eta = 0.0005  # carretera
omega = 0.3  # ignición

Nx_init = 0.45
Ny_init = 0.40

# ---------------- GRID ---------------- #
Lx = 300
Ly = 300
tf = 2000

t_rate = 5

dx = 1
dy = 1
dt = 3

[tmin, tmax, dt] = [0, tf, dt]
[xmin, xmax, dx] = [- Lx / 2,  Lx / 2 + dx, dx]
[ymin, ymax, dy] = [- Ly / 2, Ly / 2 + dy, dy]

t_grid = np.arange(tmin, tmax + dt, dt)
x_grid = np.arange(xmin, xmax, dx)
y_grid = np.arange(ymin, ymax, dy)

Nt = t_grid.shape[0]
Nx = x_grid.shape[0]
Ny = y_grid.shape[0]

X, Y = np.meshgrid(x_grid, y_grid)

######################## PRE-PROCESSING FOR SIMULATION ########################

# ---------------- HETEROGENEOS FUEL DISTRIBUTION ----------------
np.random.seed(47)
chi = np.random.rand(Ny, Nx)   # ojo: Ny, Nx (consistente con U)
chi = chi ** 1.5  # empuja la masa hacia 0
chi = gaussian_filter(chi, sigma=5)
chi = (chi - chi.min()) / (chi.max() - chi.min())

sea_mask = np.zeros((Ny, Nx), dtype=bool)
sea_mask[:, :int(0.2 * Nx)] = True
chi[sea_mask] = 0.0

# ---------------- INITIAL CONDITIONS ----------------
U_1_init = np.zeros((Ny, Nx))
U_2_init = np.ones((Ny, Nx)) + 2.0 * (chi - 0.5)
U_2_init[:, :int(0.2*len(x_grid))] = 0.0

I_init = int(Nx_init * Nx)
J_init = int(Ny_init * Ny)
U_1_init[J_init-2:J_init+2, I_init-2:I_init+2] = 1

In [23]:
# ---------------- OPERATORS ----------------
#### ROAD KERNEL ####
N = Nx * Ny
S = 5
ell = 0.2
K_road = lil_matrix((N, N))
x_road = Nx // 2

for iy in range(Ny):
    if sea_mask[iy, x_road]:
        continue
    i = iy * Nx + x_road
    for offset in range(-S, S + 1):
        if offset == 0:
            continue
        iy2 = iy + offset
        if iy2 < 0 or iy2 >= Ny:
            continue
        if sea_mask[iy2, x_road]:
            continue
        j = iy2 * Nx + x_road
        K_road[i, j] = np.exp(-abs(offset) / ell)

#### DIFFUSION LAPLACIAN ####
D2x = sparse_DD_neumann(Nx, dx)
D2y = sparse_DD_neumann(Ny, dy)
Ix = identity(Nx)
Iy = identity(Ny)
L = kron(Iy, D2x) + kron(D2y, Ix)

# ----------------  PACKING FOR SIMULATION ----------------
U_1_flat = U_1_init.reshape(-1)
U_2_flat = U_2_init.reshape(-1)
fields_init = np.array([U_1_flat, U_2_flat])
grids = [t_grid, x_grid, y_grid]
parameters = {"omega": omega, "beta": beta, "gamma": gamma, "alpha": alpha, "delta": delta, "eta": eta}
operators = [L, K_road]

In [26]:
######################## INTEGRACIÓN NUMÉRICA ########################

final_fields, fields_history, time_grid = RK4_FD(eq, fields_init, parameters, grids, dt, Nt, operators, t_rate)

Hora de Inicio: 17:33:26
Hora de Término: 17:33:53
27.094496250152588 seg


In [27]:
######################## SAVING DATA AND PLOTS ########################
fields_history = np.array(fields_history)
U1_light = fields_history[:, 0].reshape(-1, Ny, Nx)
U2_light = fields_history[:, 1].reshape(-1, Ny, Nx)

beta_str = f"{beta:.4f}"
gamma_str = f"{gamma:.4f}"
delta_str = f"{delta:.4f}"
alpha_str = f"{alpha:.4f}"
eta_str = f"{eta:.4f}"
omega_str = f"{omega:.4f}"

file = disc + 'simulation_data' + project_name
subfile = "/beta=" + beta_str + "/gamma=" + gamma_str + "/delta=" + delta_str + "/alpha=" + alpha_str + "/eta=" + eta_str + "/omega=" + omega_str
if not os.path.exists(file + subfile):
    os.makedirs(file + subfile)

# ---------------- ANIMATIONS ----------------

sea_overlay = sea_mask.astype(float)
colors = [(0.0, "#3b0101"), (0.02, "#a80300"), (0.04, "#5cbd02"), (0.5, "#249103"), (1.0, "#124701")]
cmap_custom = LinearSegmentedColormap.from_list("custom_RdYlGn", colors)

fig, ax = plt.subplots(figsize=(5, 4))

ax.set(xlim=(xmin, xmax), ylim=(ymin, ymax))
ax.set_aspect('equal')
cax = ax.pcolormesh(x_grid, y_grid, U1_light[0], cmap="OrRd", shading='auto')   # TEMPERARURE
sea = ax.pcolormesh(x_grid, y_grid, sea_overlay, cmap='Blues', shading='auto')  # WATER
sea.set_alpha(sea_overlay)
x_line = x_grid[x_road]
ax.plot(np.ones_like(y_grid) * x_line, y_grid, 'k-', linewidth=2, alpha=0.3)
cbar = fig.colorbar(cax)
cbar.set_label('$T(\\vec{x})$', rotation=0, size=20, labelpad=-27, y=1.13)
def animate(i):
    cax.set_array(U1_light[i].ravel())

anim = FuncAnimation(fig, animate, interval=100, frames=len(U1_light)-1)
anim.save(file + subfile + '/T.gif', writer=animation.FFMpegWriter(fps=10), dpi=300)
plt.close()

fig, ax = plt.subplots(figsize=(5, 4))
ax.set(xlim=(xmin, xmax), ylim=(ymin, ymax))
ax.set_aspect('equal')
cax = ax.pcolormesh(x_grid, y_grid, U2_light[0], cmap=cmap_custom, shading='auto', vmin=0, vmax=np.amax(U2_light[0]))
sea = ax.pcolormesh(x_grid, y_grid, sea_overlay, cmap='Blues', shading='auto')
sea.set_alpha(sea_overlay)
x_line = x_grid[x_road]
ax.plot(np.ones_like(y_grid) * x_line, y_grid, 'k-', linewidth=2, alpha=0.3)

cbar = fig.colorbar(cax)
cbar.set_label('$Y(\\vec{x})$', rotation=0, size=20, labelpad=-27, y=1.13)

def animate(i):
    cax.set_array(U2_light[i].ravel())
anim = FuncAnimation(fig, animate, interval=100, frames=len(U2_light)-1)
anim.save(file + subfile + '/Y.gif', writer=animation.FFMpegWriter(fps=10), dpi=300)
plt.close()